# 🤖 AdrIA - Academia de IA con Descarga de PDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IAparatodos/Web-gemini-3/blob/claude/fix-pdf-image-sizing-012HXq1ZE91CVMMhq18oPWkk/Run_AdrIA_in_Colab.ipynb)

Este notebook ejecuta la aplicación web AdrIA en Google Colab.

**Funcionalidades:**
- Chat con AdrIA (IA conversacional con Gemini)
- Descarga de instrucciones en PDF con imágenes grandes
- Interfaz moderna con React + Tailwind CSS

---

## 📝 Instrucciones
1. Ejecuta las celdas en orden (Ctrl+Enter o botón ▶️)
2. Espera a que aparezca la URL pública (ngrok)
3. Haz clic en la URL para abrir la aplicación
4. ¡Prueba el botón de descarga de PDF!

## 1️⃣ Instalar Node.js y dependencias

In [ ]:
%%bash
# Instalar Node.js 20.x
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs

# Verificar instalación
node --version
npm --version

## 2️⃣ Clonar repositorio desde GitHub

In [ ]:
%%bash
# Limpiar si existe
rm -rf Web-gemini-3

# Clonar el repositorio
git clone https://github.com/IAparatodos/Web-gemini-3.git
cd Web-gemini-3

# Cambiar a la rama con la funcionalidad de PDF
git checkout claude/fix-pdf-image-sizing-012HXq1ZE91CVMMhq18oPWkk

# Mostrar información
echo "✅ Repositorio clonado"
git log --oneline -3

## 3️⃣ Configurar API Key de Gemini

**⚠️ IMPORTANTE:** Necesitas una API key de Google AI Studio

1. Ve a: https://aistudio.google.com/app/apikey
2. Crea una API key
3. Pégala en la celda de abajo

In [ ]:
import os

# 👇 PEGA TU API KEY AQUÍ (entre las comillas)
GEMINI_API_KEY = "TU_API_KEY_AQUI"

# Crear archivo .env.local
with open('/content/Web-gemini-3/.env.local', 'w') as f:
    f.write(f'VITE_GEMINI_API_KEY={GEMINI_API_KEY}\n')

print("✅ API Key configurada")

## 4️⃣ Instalar dependencias del proyecto

In [ ]:
%%bash
cd /content/Web-gemini-3
npm install
echo "✅ Dependencias instaladas"

## 5️⃣ Instalar y configurar ngrok (para URL pública)

In [ ]:
%%bash
# Descargar ngrok
wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
tar -xzf ngrok-v3-stable-linux-amd64.tgz
chmod +x ngrok
mv ngrok /usr/local/bin/
echo "✅ ngrok instalado"

## 5️⃣-B Configurar ngrok authtoken

**⚠️ IMPORTANTE:** ngrok requiere un token gratuito

1. Ve a: https://dashboard.ngrok.com/signup (crea cuenta gratis)
2. Copia tu authtoken de: https://dashboard.ngrok.com/get-started/your-authtoken
3. Pégalo en la celda de abajo

In [ ]:
import subprocess

# 👇 PEGA TU NGROK AUTHTOKEN AQUÍ (entre las comillas)
NGROK_AUTHTOKEN = "TU_NGROK_AUTHTOKEN_AQUI"

# Configurar authtoken
result = subprocess.run(
    ['ngrok', 'config', 'add-authtoken', NGROK_AUTHTOKEN],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ ngrok authtoken configurado correctamente")
else:
    print("❌ Error al configurar authtoken:")
    print(result.stderr)

## 6️⃣ Iniciar aplicación

**⚠️ Esta celda no terminará de ejecutarse**

Aparecerá una URL pública (ngrok) que debes abrir en tu navegador.

In [ ]:
import subprocess
import time
import requests
from threading import Thread
import sys

def run_vite():
    """Ejecutar servidor de desarrollo Vite"""
    print("🔧 Iniciando servidor Vite...")
    process = subprocess.Popen(
        ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '3000'],
        cwd='/content/Web-gemini-3',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    # Mostrar output de Vite
    for line in process.stdout:
        if 'Local:' in line or 'ready in' in line or 'error' in line.lower():
            print(f"  Vite: {line.strip()}")

def run_ngrok():
    """Ejecutar ngrok y mostrar URL pública"""
    print("⏳ Esperando a que Vite inicie (10 segundos)...")
    time.sleep(10)
    
    print("🌐 Iniciando túnel ngrok...")
    # Iniciar ngrok
    process = subprocess.Popen(
        ['ngrok', 'http', '3000', '--log', 'stdout'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Esperar más tiempo para que ngrok se inicie
    print("⏳ Esperando a que ngrok se conecte (8 segundos)...")
    time.sleep(8)
    
    # Obtener URL pública con reintentos
    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = requests.get('http://localhost:4040/api/tunnels', timeout=5)
            data = response.json()
            
            if 'tunnels' in data and len(data['tunnels']) > 0:
                public_url = data['tunnels'][0]['public_url']
                
                print("\n" + "="*70)
                print("🚀 ¡APLICACIÓN CORRIENDO EXITOSAMENTE!")
                print("="*70)
                print(f"\n📱 Abre esta URL en tu navegador:\n")
                print(f"   {public_url}")
                print("\n" + "="*70)
                print("\n✨ Funcionalidades disponibles:")
                print("   • Chat con AdrIA (IA conversacional con Gemini)")
                print("   • Botón 'Descargar Instrucciones PDF' en panel izquierdo")
                print("   • PDF con imágenes GRANDES (no pequeñas)")
                print("\n⚠️  Para detener: Menú Runtime → Interrupt execution")
                print("="*70 + "\n")
                return
            else:
                print(f"⏳ Intento {attempt + 1}/{max_retries}: Esperando túneles...")
                time.sleep(3)
                
        except requests.exceptions.RequestException as e:
            print(f"⏳ Intento {attempt + 1}/{max_retries}: Esperando API de ngrok...")
            time.sleep(3)
        except Exception as e:
            print(f"⚠️  Error inesperado: {e}")
            time.sleep(3)
    
    # Si llegamos aquí, no pudimos obtener la URL
    print("\n" + "="*70)
    print("❌ No se pudo obtener la URL de ngrok")
    print("="*70)
    print("\n🔍 Posibles causas:")
    print("   1. No configuraste el NGROK_AUTHTOKEN (vuelve a la celda 5️⃣-B)")
    print("   2. El authtoken es inválido")
    print("   3. Problemas de red")
    print("\n💡 Soluciones:")
    print("   1. Verifica que copiaste bien el authtoken de:")
    print("      https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   2. Ejecuta la celda 5️⃣-B nuevamente con el token correcto")
    print("   3. Vuelve a ejecutar esta celda (6️⃣)")
    print("="*70 + "\n")

# Iniciar ambos procesos
print("="*70)
print("🚀 INICIANDO ADRIA EN GOOGLE COLAB")
print("="*70 + "\n")

vite_thread = Thread(target=run_vite, daemon=True)
vite_thread.start()

run_ngrok()

# Mantener vivo
print("✅ Servidor activo. Presiona el botón ⬛ (stop) para detener.\n")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Servidor detenido")

---

## 📚 Notas adicionales

### ⚠️ Requisitos importantes:
1. **API Key de Gemini** - Obtén una gratis en: https://aistudio.google.com/app/apikey
2. **ngrok authtoken** - Obtén uno gratis en: https://dashboard.ngrok.com/get-started/your-authtoken

### Archivos importantes:
- `services/pdfService.ts` - Servicio de generación/descarga de PDFs
- `components/AiAssistant.tsx` - Componente del chat con botón de descarga
- `public/Instrucciones proveedor.pdf` - PDF con imágenes grandes (376 KB)
- `vite.config.ts` - Configuración de Vite (ya incluye soporte para ngrok)

### 🔧 Troubleshooting:

**Error: "No se pudo obtener la URL de ngrok"**
- ✅ Verifica que configuraste el NGROK_AUTHTOKEN en la celda 5️⃣-B
- ✅ Copia el token correcto de: https://dashboard.ngrok.com/get-started/your-authtoken
- ✅ Vuelve a ejecutar la celda 5️⃣-B y luego la celda 6️⃣

**Error: "This host is not allowed" (ngrok)**
- ✅ Ya está solucionado en el repositorio (vite.config.ts incluye allowedHosts)
- ✅ Si clonaste antes del fix, vuelve a ejecutar la celda 2️⃣ (clonar repo)
- ✅ El archivo vite.config.ts ya permite todos los dominios de ngrok

**La URL no carga:**
- Espera 10-15 segundos y recarga la página
- Verifica que la celda 6️⃣ sigue ejecutándose (no la detengas)

**El PDF no descarga:**
- Verifica que el archivo esté en `/content/Web-gemini-3/public/`
- Ejecuta: `!ls -lh /content/Web-gemini-3/public/*.pdf`

**Para ver logs detallados:**
- Revisa la salida de las celdas anteriores
- Los mensajes de Vite aparecerán durante el inicio

### 📤 Subir tus propios PDFs:
1. Usa el panel de archivos 📁 en Colab
2. Navega a `/content/Web-gemini-3/public/`
3. Sube tu PDF
4. Actualiza `services/pdfService.ts` con el nuevo nombre

### 🌐 Repositorio:
- **GitHub:** https://github.com/IAparatodos/Web-gemini-3
- **Rama actual:** `claude/fix-pdf-image-sizing-012HXq1ZE91CVMMhq18oPWkk`

### 💡 Tecnologías usadas:
- React + Vite + TypeScript
- Tailwind CSS
- Google Gemini AI
- jsPDF (generación de PDFs)
- ngrok (túnel público)